In [1]:
import os
import pandas as pd
import numpy as np
from transformers import BertTokenizerFast, BertModel
import torch
from sklearn.metrics.pairwise import cosine_distances
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()

/Users/pedropertusi/Desktop/contentVsForm/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Importing Rewrites

### SAT Rewrites 

In [2]:
sat_rewrites_dir = "../data/rewrites/sat_full/cleaned/"

sat_rew = []
for i in range(1, 7):
    fname = f"rew_sat_{i}.csv"
    path = os.path.join(sat_rewrites_dir, fname)
    df = pd.read_csv(path)
    sat_rew.append(df)

### No Description Rewrite

In [3]:
no_desc_rewrites_dir = "../data/rewrites/no_desc/cleaned/no_desc_rewritten_0_cleaned.csv"
df_no_desc = pd.read_csv(no_desc_rewrites_dir)

/var/folders/lm/psfj50g95wgczw9z2l6zf32m0000gn/T/ipykernel_31230/3774473763.py:2: DtypeWarning: Columns (375) have mixed types. Specify dtype option on import or set low_memory=False.
  df_no_desc = pd.read_csv(no_desc_rewrites_dir)


----

### Data Analysis on content preserved flaggin in rewrites

##### SAT Rewrites Analysis

In [ ]:
for i,rew in enumerate(sat_rew):
    print(f"sat {i+1}, content preserved - True: {rew['content_preserved'].value_counts().iloc[0]} False: {rew['content_preserved'].value_counts().iloc[1]}")

In [ ]:
mean_not_preserved = sum([x["content_preserved"].value_counts().iloc[1] for x in sat_rew])/len(sat_rew)
print(f"Mean content not preserved SAT Rewrites - {mean_not_preserved:.2f}")

##### No Description Analysis

In [ ]:
no_desc_rewrites_dir = "../data/rewrites/no_desc/cleaned/no_desc_rewritten_0_cleaned.csv"
df_no_desc = pd.read_csv(no_desc_rewrites_dir)

In [ ]:
print(
    f"no_desc — "
    f"True: {df_no_desc['content_preserved_0'].value_counts().iloc[0]}, False: {df_no_desc['content_preserved_0'].value_counts().iloc[1]}")

-----

### Embeddings distance OG x Rewrites

In [4]:
device = torch.device('mps') if (torch.backends.mps.is_available()) else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').eval().to(device)

def get_embeddings(text: str) -> np.ndarray:
    """
    Run one text through BERT, return the [CLS] embedding as a numpy vector.
    """
    inputs = tokenizer(text,
                       return_tensors='pt',
                       padding=True,
                       truncation=True,
                       max_length=512)
    # move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # [batch=1, seq, dim] → pick CLS token embedding
    cls_emb = outputs.last_hidden_state[0, 0, :]
    return cls_emb.cpu().numpy()

def embed_text(text):
    if text is None:
        text = ""
    emb = get_embeddings(text)
    return np.asarray(emb).reshape(1, -1)

In [ ]:
# Collect all unique texts we will ever embed
all_texts = set()

# SAT rewrites
for df in sat_rew:
    all_texts.update(df["text"].dropna().unique())
    all_texts.update(df["rewritten_text"].dropna().unique())

# No-description rewrites
all_texts.update(df_no_desc["text"].dropna().unique())
all_texts.update(df_no_desc["rewritten_text_0"].dropna().unique())

all_texts = list(all_texts)

In [ ]:
# Embed once
text2emb = {}
for t in tqdm(all_texts):
    text2emb[t] = embed_text(t).reshape(-1)  # (d,)

----

### Paralelism

In [ ]:
from tqdm import tqdm
import math

def embed_texts_batch(texts, batch_size=16):
    """
    Embed a list of texts using BERT in batches, with tqdm progress bar.
    Returns a list of embeddings aligned with `texts`.
    """
    embeddings = []

    n_batches = math.ceil(len(texts) / batch_size)

    for i in tqdm(
        range(0, len(texts), batch_size),
        total=n_batches,
        desc="Embedding texts",
        unit="batch"
    ):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # CLS token embeddings: [batch, dim]
        cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.extend(cls_embs)

    return embeddings

In [ ]:
all_texts = list(all_texts)

embs = embed_texts_batch(all_texts, batch_size=16)

text2emb = dict(zip(all_texts, embs))

---

In [ ]:
np.savez_compressed(
    "bert_cls_embeddings.npz",
    **text2emb
)

In [5]:
loaded = np.load("bert_cls_embeddings.npz", allow_pickle=True)
text2emb = {k: loaded[k] for k in loaded.files}

In [7]:
# Build mapping: original text -> no-description rewritten text
df_no_desc_clean = df_no_desc.dropna(
    subset=["text", "rewritten_text_0"]
).copy()

# If there are duplicates, keep the first occurrence
no_desc_map = (
    df_no_desc_clean
    .drop_duplicates("text")
    .set_index("text")["rewritten_text_0"]
    .to_dict()
)


In [8]:
from sklearn.metrics.pairwise import cosine_distances

results = []

B = 3  # number of random baselines per original

rew_0 = sat_rew[0]  # originals live here

for idx, row in rew_0.iterrows():

    og_text = row["text"]
    prompt  = row["prompt_name"]
    ses     = row["economically_disadvantaged"]

    # original embedding
    if og_text not in text2emb:
        continue
    og_emb = text2emb[og_text].reshape(1, -1)

    # --------------------------------------------------
    # 1) original → SAT rewrites (1–6)
    # --------------------------------------------------
    for rewrite_id, r_df in enumerate(sat_rew, start=1):

        sub = r_df[r_df["text"] == og_text]
        if len(sub) == 0:
            continue

        rew_text = sub["rewritten_text"].iloc[0]
        if rew_text not in text2emb:
            continue

        rew_emb = text2emb[rew_text].reshape(1, -1)
        d_og_rew = cosine_distances(og_emb, rew_emb)[0, 0]

        results.append({
            "text_id": idx,
            "type": "rewrite",
            "rewrite_id": rewrite_id,
            "distance": d_og_rew,
            "prompt": prompt,
            "ses": ses
        })

    # --------------------------------------------------
    # 2) original → NO-DESCRIPTION rewrite
    # --------------------------------------------------
    no_desc_text = no_desc_map.get(og_text)

    if no_desc_text is not None and no_desc_text in text2emb:
        no_desc_emb = text2emb[no_desc_text].reshape(1, -1)
        d_og_no_desc = cosine_distances(og_emb, no_desc_emb)[0, 0]

        results.append({
            "text_id": idx,
            "type": "no_desc_rewrite",
            "rewrite_id": None,
            "distance": d_og_no_desc,
            "prompt": prompt,
            "ses": ses
        })

    # --------------------------------------------------
    # 3) original → B random originals (same SES + prompt)
    # --------------------------------------------------
    pool = rew_0[
        (rew_0["prompt_name"] == prompt) &
        (rew_0["economically_disadvantaged"] == ses) &
        (rew_0["text"] != og_text)
    ]

    if len(pool) == 0:
        continue

    sampled = pool.sample(
        n=min(B, len(pool)),
        replace=False,
        random_state=42
    )

    for _, srow in sampled.iterrows():

        rand_text = srow["text"]
        if rand_text not in text2emb:
            continue

        rand_emb = text2emb[rand_text].reshape(1, -1)
        d_og_rand = cosine_distances(og_emb, rand_emb)[0, 0]

        results.append({
            "text_id": idx,
            "type": "random_same_ses_prompt",
            "rewrite_id": None,
            "distance": d_og_rand,
            "prompt": prompt,
            "ses": ses
        })

In [9]:
df_dist = pd.DataFrame(results)

df_dist["type"].value_counts()

type
rewrite                   119880
random_same_ses_prompt     59940
no_desc_rewrite            19966
Name: count, dtype: int64

In [10]:
# default: random baseline
df_dist["comparison"] = "Random (Same SES, Prompt)"

# SAT rewrites
mask_rew = df_dist["type"] == "rewrite"
df_dist.loc[mask_rew, "comparison"] = (
    "Rewrite " + df_dist.loc[mask_rew, "rewrite_id"].astype(int).astype(str)
)

# no-description rewrite
df_dist.loc[
    df_dist["type"] == "no_desc_rewrite",
    "comparison"
] = "No-Description Rewrite"

In [11]:
df_collapsed = (
    df_dist
    .groupby(["text_id", "comparison"])
    .agg(distance=("distance", "mean"))
    .reset_index()
)


In [21]:
def bootstrap_se(
    df,
    group_col="comparison",
    id_col="text_id",
    value_col="distance",
    B=1000,
    seed=42
):
    rng = np.random.default_rng(seed)
    boot_means = {g: [] for g in df[group_col].unique()}

    # unique essays
    text_ids = df[id_col].unique()
    n = len(text_ids)

    for _ in range(B):
        # sample essays WITH replacement
        sampled_ids = rng.choice(text_ids, size=n, replace=True)

        # rebuild bootstrap sample with multiplicity
        boot_df = (
            pd.DataFrame({id_col: sampled_ids})
            .merge(df, on=id_col, how="left")
        )

        for g in boot_means:
            boot_means[g].append(
                boot_df.loc[boot_df[group_col] == g, value_col].mean()
            )

    return {
        g: np.std(boot_means[g], ddof=1)
        for g in boot_means
    }

In [ ]:
boot_se = bootstrap_se(df_collapsed, B=1000)

summary = (
    df_collapsed
    .groupby("comparison")
    .agg(mean=("distance", "mean"))
    .reset_index()
)

summary["se"] = summary["comparison"].map(boot_se)

,comparison,mean,se
0,No-Description Rewrite,0.139871,0.000503
1,"Random (Same SES, Prompt)",0.175990,0.000313
2,Rewrite 1,0.198479,0.000647
3,Rewrite 2,0.130018,0.000394
4,Rewrite 3,0.095323,0.000378
5,Rewrite 4,0.102472,0.000440
6,Rewrite 5,0.179573,0.000587
7,Rewrite 6,0.204073,0.000611


In [31]:
table = (
    summary
    .assign(
        mean_se=lambda d: [
            f"{m:.3f} ({s:.3f})"
            for m, s in zip(d["mean"], d["se"])
        ]
    )
    .set_index("comparison")[["mean_se"]]
    .rename(columns={"mean_se": "Mean (SE)"})
    .rename_axis(None) 
)

table

,Mean (SE)
No-Description Rewrite,0.140 (0.001)
"Random (Same SES, Prompt)",0.176 (0.000)
Rewrite 1,0.198 (0.001)
Rewrite 2,0.130 (0.000)
Rewrite 3,0.095 (0.000)
Rewrite 4,0.102 (0.000)
Rewrite 5,0.180 (0.001)
Rewrite 6,0.204 (0.001)


In [33]:
df_dist.to_csv("../tables/analysis/cosine_dist_raw.csv", index=False)
df_collapsed.to_csv("../tables/analysis/cosine_dist_collapsed.csv", index=False)

table.to_latex(
    "../tables/analysis/cosine_distance_comparison.tex",
)

----

### Data Analysis Prompt Names and Assignments

In [ ]:
full_df = pd.read_csv("../data/persuade/persuade_2.0_human_scores_demo_id_github.csv")

In [ ]:
full_df['prompt_name'].value_counts()

In [ ]:
full_df['assignment'].value_counts()